# 📓 Semana 7 · Dia 3 — Linhagem, tags e auditoria com system tables

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (parcial) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (governança) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Linhagem documentada + query de auditoria |

---


## 📖 Teoria — Linhagem de dados

A **linhagem** mostra de onde veio cada coluna/tabela (origem → transformação → destino). No Databricks: **Catalog → tabela → Lineage**. Ajuda em: impacto de mudança, auditoria, confiança nos dados.

**Tags** classificam objetos (ex.: `PII`, `LGPD`, `região`).


## 📖 Teoria — Auditoria e system tables

O **system tables** (`system.access.audit`) registra quem acessou o quê, quando e de onde. Em produção (paga), fica no catálogo `system.*`:

```sql
SELECT * FROM system.access.audit
WHERE action_name = 'QUERY' AND event_date = current_date()
```

Na Free Edition, o acesso ao `system.*` é limitado — estude a estrutura e use a linhagem da UI.


### 💻 Na prática — Tags e comentários

Classifique os objetos do projeto.


In [ ]:
%sql
-- Tags de classificação
ALTER TABLE workspace.bronze.vendas_bronze SET TAGS ('PII' = 'true', 'LGPD' = 'true');
ALTER TABLE workspace.ouro.vendas_por_dia SET TAGS ('BI' = 'true');
SHOW TAGS ON TABLE workspace.bronze.vendas_bronze;

In [ ]:
%sql
-- Comentários (documentação viva)
COMMENT ON TABLE workspace.bronze.vendas_bronze IS 'Bronze de vendas — dataset Online Retail (UCI / Databricks samples)';
DESCRIBE TABLE EXTENDED workspace.bronze.vendas_bronze;

### 💻 Na prática — Linhagem pela UI

1. **Catalog → workspace.bronze.vendas_bronze → tab Lineage**.
2. Veja: vendas_bronze → fato_vendas → vendas_por_dia.
3. Clique numa coluna: linhagem em nível de coluna.


In [ ]:
# Exemplo de query de auditoria (produção paga)
query_auditoria = """
SELECT user_identity.email, action_name, request_params.path,
       event_time
FROM system.access.audit
WHERE event_date >= current_date() - INTERVAL 7 DAY
ORDER BY event_time DESC
LIMIT 20
"""
print(query_auditoria)
print("Na Free, use a UI (Audit logs não expostos por completo).")

> 🎯 **Dica de prova**: Pergunta DEP: 'como descobrir quem acessou uma tabela?' → system.access.audit. 'De onde veio essa coluna?' → linhagem. 'O que é esse dado?' → tags/comentários.


## 🎯 Exercícios de fixação

**1.** Classifique 3 tabelas do seu projeto com tags.

**2.** Use a linhagem da UI para desenhar o fluxo de vendas_bronze → Ouro.

**3.** Escreva a query de auditoria que lista os últimos 10 acessos a uma tabela.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Tags

`ALTER TABLE ... SET TAGS ('PII'='true')` — classificação para governança e busca.

**2.** Linhagem

Catalog → tabela → Lineage: veja upstream (bronze) e downstream (prata/ouro).

**3.** Auditoria

`SELECT user_identity.email, action_name, event_time FROM system.access.audit WHERE request_params.path LIKE '%vendas%' ORDER BY event_time DESC LIMIT 10`.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*